In [ ]:
import polars as pl
from pathlib import Path
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

import msoffcrypto
from io import BytesIO

import gc
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [ ]:
# Configure Polars display settings
pl.Config.set_tbl_cols(-1)  # Limit to 10 columns
pl.Config.set_tbl_rows(-1)  # Limit to 20 rows
pl.Config.set_tbl_width_chars(200)  # Set the width of columns to 25 characters
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dtype_separator(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_cell_alignment("RIGHT")
pl.Config.float_precision=3

gc.collect()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Snowflake connection details
account = 'tata.us-east-1'
user = ''
password = ''
database = 'PROD'
schema = 'MQA_AMPLIFY_ABM'
warehouse = 'BSM_QA_WAREHOUSE_MEDIUM'
role = 'BSM_DEVELOPER'

# Establish connection
conc = snowflake.connector.connect(
            account=account,
            user=user,
            password=password,
            warehouse=warehouse,
            DATABASE=database,   
            SCHEMA=schema,      
            role=role)



In [ ]:
db_list = ['ADP_PUBLISH','BSD_PUBLISH','RAW','BSM_ETL','ADP_WORKSPACES','PROD', 'EIO_PUBLISH']

query = """
SELECT t.TABLE_CATALOG, t.table_schema, t.table_name, column_name
FROM {db_nm}.information_schema.tables t
INNER JOIN {db_nm}.information_schema.columns c ON c.table_schema = t.table_schema AND c.table_name = t.table_name
WHERE UPPER(column_name) LIKE '%ACTION%' AND UPPER(t.table_schema) LIKE ANY ('%SFDC%') 
ORDER BY t.table_schema, t.table_name, column_name;
"""

In [ ]:
all_results = []

for db_nm in db_list:
    
    cursr = conc.cursor()

    try:
        cursr.execute(query.format(db_nm=db_nm))
        print(f"{'*' * 10} Table {db_nm} successfully.")
    except Exception as e:
        print(f"{'*' * 10} Error: {db_nm}: {e}")

    results = cursr.fetchall()
    df = pl.DataFrame(results, schema={"TABLE_CATALOG": pl.Utf8, "table_schema": pl.Utf8, "table_name": pl.Utf8, "column_name": pl.Utf8})

    all_results.append(df)

    cursr.close()

conc.close()

combined_df = pl.concat(all_results)

In [ ]:
# display(combined_df)

df_pandas = combined_df.to_pandas()

# Write the Pandas DataFrame to an Excel file
df_pandas.to_excel("action_id_columns.xlsx", index=False)